<a href="https://colab.research.google.com/github/Lolla-data-analyst/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
%pip install -q duckdb huggingface_hub

In [2]:
import duckdb
from google.colab import userdata

con = duckdb.connect()
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{userdata.get('HF_TOKEN')}')")

print("Connected successfully")

Connected successfully


In [3]:
con.execute(f"CREATE OR REPLACE SECRET (TYPE huggingface, TOKEN '{userdata.get('HF_TOKEN')}')")

In [4]:
rel = "hf://datasets/FlyRank/internship-warehouse"

test = con.sql(f"SELECT * FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet') LIMIT 5")
test.show()

┌─────────────┬─────────────────────────┬──────────────────────────┬────────────────┬────────────────┬────────────────────┬────────────────────┬─────────────────┬────────────┬──────────────────┬────────────────────┬───────────────┬──────────────┬───────────┬──────────────────────┬──────────────────────────┬──────────────────┬─────────────────┬───────────────────┬─────────────────┬───────────────┬─────────────┬────────────┬───────────────┬───────────┬────────────┬───────────┬─────────┬──────────┬───────────────┬─────────┐
│ report_date │     client_hash_id      │     content_hash_id      │ client_has_gsc │ client_has_ga4 │ gsc_data_available │ ga4_data_available │ gsc_impressions │ gsc_clicks │ gsc_sum_position │  gsc_avg_position  │ ga4_pageviews │ ga4_sessions │ ga4_users │ ga4_engaged_sessions │ ga4_total_engagement_sec │ sessions_organic │ sessions_direct │ sessions_referral │ sessions_social │ sessions_paid │ sessions_ai │ ai_chatgpt │ ai_perplexity │ ai_gemini │ ai_copilot │ ai_cl

In [5]:
result = con.sql(f"""
SELECT report_date, client_hash_id, content_hash_id, COUNT(*) AS row_count
FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')
WHERE report_date >= '2026-03-01' AND report_date < '2026-04-01'
GROUP BY report_date, client_hash_id, content_hash_id
ORDER BY row_count DESC
LIMIT 10
""")
result.show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌─────────────┬─────────────────────────┬──────────────────────────┬───────────┐
│ report_date │     client_hash_id      │     content_hash_id      │ row_count │
│    date     │         varchar         │         varchar          │   int64   │
├─────────────┼─────────────────────────┼──────────────────────────┼───────────┤
│ 2026-03-01  │ client_62f4a7e64f5e0096 │ content_d0dff76c889de68f │         1 │
│ 2026-03-01  │ client_62f4a7e64f5e0096 │ content_67741cce996cfafa │         1 │
│ 2026-03-01  │ client_62f4a7e64f5e0096 │ content_2e6360ad20fd7107 │         1 │
│ 2026-03-01  │ client_62f4a7e64f5e0096 │ content_ac8663da7484669a │         1 │
│ 2026-03-01  │ client_62f4a7e64f5e0096 │ content_65c50dfe9d87a585 │         1 │
│ 2026-03-01  │ client_62f4a7e64f5e0096 │ content_d49a012dcb924e31 │         1 │
│ 2026-03-01  │ client_62f4a7e64f5e0096 │ content_614baf2af4330bd7 │         1 │
│ 2026-03-01  │ client_62f4a7e64f5e0096 │ content_4dc944b7d0b65ecc │         1 │
│ 2026-03-01  │ client_62f4a

In [6]:
result = con.sql(f"""
SELECT MIN(report_date), MAX(report_date), Count(*) AS row_count
FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')
WHERE report_date >= '2026-03-01' AND report_date < '2026-04-01' AND gsc_data_available = true
""")
result.show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌──────────────────┬──────────────────┬───────────┐
│ min(report_date) │ max(report_date) │ row_count │
│       date       │       date       │   int64   │
├──────────────────┼──────────────────┼───────────┤
│ 2026-03-01       │ 2026-03-31       │   3611061 │
└──────────────────┴──────────────────┴───────────┘



In [7]:
result = con.sql(f"""
SELECT COUNT(*) AS total_rows,
COUNT(CASE WHEN gsc_data_available = true THEN 1 ELSE NULL END) AS available_rows
FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')
WHERE report_date >= '2026-03-01' AND report_date < '2026-04-01'
""")
result.show()

┌────────────┬────────────────┐
│ total_rows │ available_rows │
│   int64    │     int64      │
├────────────┼────────────────┤
│    9841378 │        3611061 │
└────────────┴────────────────┘



In [8]:
result = con.sql(f"""
SELECT CORR(gsc_avg_position, gsc_sum_position)
FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')
WHERE report_date >= '2026-03-01' AND report_date < '2026-04-01'
""")
result.show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌──────────────────────────────────────────┐
│ corr(gsc_avg_position, gsc_sum_position) │
│                  double                  │
├──────────────────────────────────────────┤
│                      0.11613006030170202 │
└──────────────────────────────────────────┘



In [9]:
result = con.sql(f"""
SELECT MAX(ABS(gsc_avg_position - gsc_sum_position / NULLIF(gsc_impressions, 0))) AS max_difference
FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')
WHERE report_date >= '2026-03-01' AND report_date < '2026-04-01'
""")
result.show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────────┐
│ max_difference │
│     double     │
├────────────────┤
│            0.0 │
└────────────────┘



The trap: I deliberately considered adding gsc_sum_position as a feature. A correlation check alone was misleading (only 0.116, since impressions vary independently), but a direct check proved the real story: gsc_sum_position ÷ gsc_impressions equals gsc_avg_position exactly, for every row in the slice (max difference = 0.0). This means gsc_sum_position is mathematically derived from the exact same information as my ranking target — including it wouldn't add real insight, it would just let the "prediction" cheat by seeing a disguised version of the answer. I excluded it and kept my honest 5-feature set instead.